# Camada Gold — Star Schema (Esquema Estrela) — Processo Sicredi

Modelo dimensional em **star schema** lendo as tabelas **particionadas** da camada Silver (`workspace.silver.*_particionado`) e gravando no schema `workspace.gold`:

## Dimensões
- **`dim_cliente`** — dados cadastrais (id_cliente, nome, cpf, cidade, estado, data_nascimento)
- **`dim_conta`** — dados da conta (id_conta, tipo_conta, status_conta, data_abertura)
- **`dim_cartao`** — dados do cartão (id_cartao, bandeira, tipo_cartao, status_cartao, data_emissao)
- **`dim_estabelecimento`** — estabelecimentos comerciais (nome, mcc, cidade)
- **`dim_tempo`** — dimensão temporal (ano, mes, trimestre, nome_mes)

## Fato
- **`fact_transacoes_financeiras`** — transações financeiras (sem estornos) com chaves surrogate (`sk_`) para todas as dimensões e medidas: `valor`, `qtd_transacao`

## Esquema Estrela
```
  dim_cliente    dim_conta    dim_cartao
       \           |             /
        fact_transacoes_financeiras
       /           |             \
  dim_estabelecimento    dim_tempo
```

In [0]:
# ============================================================
# Configuração do pipeline Gold Star Schema
# ============================================================
CATALOG = "workspace"
SILVER = f"{CATALOG}.silver"
GOLD   = f"{CATALOG}.gold"

print(f"Origem: {SILVER}.*_particionado -> Destino: {GOLD} (star schema)")

In [0]:
# ============================================================
# 1. DIMENSÕES (star schema)
# Surrogate keys via ROW_NUMBER() OVER (ORDER BY business_key)
# ============================================================

# --- dim_cliente ---
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_cliente
USING DELTA
PARTITIONED BY (ano_mes)
AS SELECT
  ROW_NUMBER() OVER (ORDER BY id_cliente) AS sk_cliente,
  id_cliente,
  nome,
  cpf,
  data_nascimento,
  cidade,
  estado,
  data_atualizacao,
  ano_mes
FROM {SILVER}.clientes_particionado
""")
print(f"dim_cliente: {spark.table(f'{GOLD}.dim_cliente').count()} registros")

# --- dim_conta ---
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_conta
USING DELTA
PARTITIONED BY (ano_mes)
AS SELECT
  ROW_NUMBER() OVER (ORDER BY id_conta) AS sk_conta,
  id_conta,
  id_cliente,
  tipo_conta,
  status_conta,
  data_abertura,
  ano_mes
FROM {SILVER}.contas_particionado
""")
print(f"dim_conta: {spark.table(f'{GOLD}.dim_conta').count()} registros")

# --- dim_cartao ---
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_cartao
USING DELTA
PARTITIONED BY (ano_mes)
AS SELECT
  ROW_NUMBER() OVER (ORDER BY id_cartao) AS sk_cartao,
  id_cartao,
  id_conta,
  bandeira,
  tipo_cartao,
  status_cartao,
  data_emissao,
  data_atualizacao,
  ano_mes
FROM {SILVER}.cartoes_particionado
""")
print(f"dim_cartao: {spark.table(f'{GOLD}.dim_cartao').count()} registros")

# --- dim_estabelecimento ---
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_estabelecimento
USING DELTA
AS SELECT
  ROW_NUMBER() OVER (ORDER BY nome_estabelecimento, mcc, cidade_estabelecimento) AS sk_estabelecimento,
  nome_estabelecimento,
  mcc,
  cidade_estabelecimento
FROM (
  SELECT DISTINCT
    nome_estabelecimento,
    mcc,
    cidade_estabelecimento
  FROM {SILVER}.transacoes_financeiras_particionado
)
""")
print(f"dim_estabelecimento: {spark.table(f'{GOLD}.dim_estabelecimento').count()} registros")

# --- dim_tempo ---
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_tempo
USING DELTA
AS SELECT
  ROW_NUMBER() OVER (ORDER BY ano_mes) AS sk_tempo,
  ano_mes,
  CAST(SUBSTRING(ano_mes, 1, 4) AS INT) AS ano,
  CAST(SUBSTRING(ano_mes, 6, 2) AS INT) AS mes,
  CASE
    WHEN CAST(SUBSTRING(ano_mes, 6, 2) AS INT) <= 3 THEN 1
    WHEN CAST(SUBSTRING(ano_mes, 6, 2) AS INT) <= 6 THEN 2
    WHEN CAST(SUBSTRING(ano_mes, 6, 2) AS INT) <= 9 THEN 3
    ELSE 4
  END AS trimestre,
  CASE CAST(SUBSTRING(ano_mes, 6, 2) AS INT)
    WHEN 1 THEN 'Jan'
    WHEN 2 THEN 'Fev'
    WHEN 3 THEN 'Mar'
    WHEN 4 THEN 'Abr'
    WHEN 5 THEN 'Mai'
    WHEN 6 THEN 'Jun'
    WHEN 7 THEN 'Jul'
    WHEN 8 THEN 'Ago'
    WHEN 9 THEN 'Set'
    WHEN 10 THEN 'Out'
    WHEN 11 THEN 'Nov'
    WHEN 12 THEN 'Dez'
  END AS nome_mes
FROM (
  SELECT DISTINCT ano_mes
  FROM {SILVER}.transacoes_financeiras_particionado
  WHERE ano_mes IS NOT NULL
)
""")
print(f"dim_tempo: {spark.table(f'{GOLD}.dim_tempo').count()} registros")

In [0]:
# ============================================================
# 2. TABELA FATO — fact_transacoes_financeiras
# Star schema: FKs diretos para todas as dimensoes (sk_)
# Base: silver.transacoes_financeiras_particionado (sem estornos)
# ============================================================

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_transacoes_financeiras
USING DELTA
PARTITIONED BY (ano_mes)
AS SELECT
  t.id_transacao,
  dc.sk_cliente,
  dc2.sk_conta,
  dc3.sk_cartao,
  de.sk_estabelecimento,
  dt.sk_tempo,
  t.data_transacao,
  t.valor,
  1 AS qtd_transacao,
  t.ano_mes
FROM {SILVER}.transacoes_financeiras_particionado t
-- cartao -> sk_cartao
JOIN {GOLD}.dim_cartao dc3
  ON t.id_cartao = dc3.id_cartao
-- conta -> sk_conta
JOIN {GOLD}.dim_conta dc2
  ON dc3.id_conta = dc2.id_conta
-- cliente -> sk_cliente
JOIN {GOLD}.dim_cliente dc
  ON dc2.id_cliente = dc.id_cliente
-- estabelecimento -> sk_estabelecimento
JOIN {GOLD}.dim_estabelecimento de
  ON COALESCE(t.nome_estabelecimento, 'N/A') = COALESCE(de.nome_estabelecimento, 'N/A')
  AND COALESCE(CAST(t.mcc AS STRING), 'N/A') = COALESCE(CAST(de.mcc AS STRING), 'N/A')
  AND COALESCE(t.cidade_estabelecimento, 'N/A') = COALESCE(de.cidade_estabelecimento, 'N/A')
-- tempo -> sk_tempo
JOIN {GOLD}.dim_tempo dt
  ON t.ano_mes = dt.ano_mes
""")

qtd_fato = spark.table(f"{GOLD}.fact_transacoes_financeiras").count()
print(f"fact_transacoes_financeiras: {qtd_fato} registros")

In [0]:
# ============================================================
# 3. VALIDAÇÃO DO STAR SCHEMA
#    - Contagem de registros
#    - Integridade referencial (FKs)
#    - Soma de valor (consistência com silver)
#    - Query exemplo (star join)
# ============================================================
from pyspark.sql import functions as F

# 3a. Resumo de registros
tabelas = [
    ("dim_cliente",                f"{GOLD}.dim_cliente"),
    ("dim_conta",                  f"{GOLD}.dim_conta"),
    ("dim_cartao",                 f"{GOLD}.dim_cartao"),
    ("dim_estabelecimento",        f"{GOLD}.dim_estabelecimento"),
    ("dim_tempo",                  f"{GOLD}.dim_tempo"),
    ("fact_transacoes_financeiras", f"{GOLD}.fact_transacoes_financeiras"),
]

linhas = [(nome, spark.table(tbl).count()) for nome, tbl in tabelas]
df_resumo = spark.createDataFrame(linhas, "tabela string, registros long")
display(df_resumo)

# 3b. Integridade referencial (registros orfaos na fato)
fato = spark.table(f"{GOLD}.fact_transacoes_financeiras")
ri_checks = []
for sk_col, dim_tbl in [
    ("sk_cliente",         f"{GOLD}.dim_cliente"),
    ("sk_conta",           f"{GOLD}.dim_conta"),
    ("sk_cartao",          f"{GOLD}.dim_cartao"),
    ("sk_estabelecimento", f"{GOLD}.dim_estabelecimento"),
    ("sk_tempo",           f"{GOLD}.dim_tempo"),
]:
    orfaos = fato.join(spark.table(dim_tbl), sk_col, "left_anti").count()
    ri_checks.append((sk_col, dim_tbl.split(".")[-1], orfaos))

df_ri = spark.createDataFrame(ri_checks, "fk_col string, dimensao string, orfaos long")
display(df_ri)

# 3c. Soma de valor (deve bater com silver)
soma_gold   = fato.agg(F.sum("valor").alias("soma")).collect()[0]["soma"]
soma_silver = spark.table(f"{SILVER}.transacoes_financeiras_particionado").agg(F.sum("valor").alias("soma")).collect()[0]["soma"]
print(f"Soma valor: gold={soma_gold} | silver={soma_silver} | match={soma_gold == soma_silver}")

# 3d. Query exemplo — star join: top 5 clientes por valor total
display(spark.sql(f"""
SELECT
  dc.nome,
  dc.estado,
  COUNT(*)      AS qtd_transacoes,
  SUM(f.valor)   AS valor_total
FROM {GOLD}.fact_transacoes_financeiras f
JOIN {GOLD}.dim_cliente dc ON f.sk_cliente = dc.sk_cliente
GROUP BY dc.nome, dc.estado
ORDER BY valor_total DESC
LIMIT 5
"""))